In [0]:
# ---------- 5. GRAVAR EM DELTA ----------
# Criar os schemas se não existirem
spark.sql("CREATE DATABASE IF NOT EXISTS silver")
spark.sql("CREATE DATABASE IF NOT EXISTS monitoring")

df = spark.createDataFrame(pdf)
df = df.drop("_ingested_at", "_source_table")

# Caminho Delta com o prefixo dbfs: (obrigatório no Databricks)
delta_path = "dbfs:/FileStore/silver/indicador_municipio"
df.write.mode("overwrite").format("delta").partitionBy("ano").save(delta_path)

# Criar tabela gerenciada para consulta
spark.sql(f"CREATE TABLE IF NOT EXISTS silver.indicador_municipio USING DELTA LOCATION '{delta_path}'")

print(f"\n[OK] Silver indicador_municipio gravada: {delta_path}")
print(f"     Registros: {df.count()} | Partições: {df.select('ano').distinct().count()}")

In [0]:
# %python
# =========================================
# SILVER — Indicador por Município
# Fonte: Bronze municipio.parquet
# Saída: silver.indicador_municipio (Delta, particionado por ano)
# =========================================
import io, sys
from pathlib import Path

repo = Path.cwd()
while not repo.name.startswith("postech-aisc") and repo.parent != repo:
    repo = repo.parent
sys.path.insert(0, str(repo))

from src.config.settings import AZURE_STORAGE_ACCOUNT, AZURE_STORAGE_KEY, BRONZE_CONTAINER
from azure.storage.blob import BlobServiceClient
import pandas as pd
from pyspark.sql import functions as F

# ---------- 1. LER DO BRONZE ----------
conn_str = (f"DefaultEndpointsProtocol=https;AccountName={AZURE_STORAGE_ACCOUNT};"
            f"AccountKey={AZURE_STORAGE_KEY};EndpointSuffix=core.windows.net")
blob_service_client = BlobServiceClient.from_connection_string(conn_str)
container_client = blob_service_client.get_container_client(BRONZE_CONTAINER)

data = container_client.get_blob_client("2026-08-30_municipio.parquet").download_blob().readall()
pdf = pd.read_parquet(io.BytesIO(data))

# ---------- 2. TRANSFORMAÇÕES ----------
# id_municipio como string de 7 dígitos (garantia)
pdf["id_municipio"] = pdf["id_municipio"].astype(str).str.zfill(7)
# Mapear rede (2=Estadual, 3=Municipal, 4=Privada) — confirmar no dicionário
rede_map = {"2": "Estadual", "3": "Municipal", "4": "Privada"}
pdf["rede_nome"] = pdf["rede"].astype(str).map(rede_map).fillna("Desconhecida")
# Metadados de rastreabilidade (princípio da arquitetura)
pdf["ingested_at"] = pdf["_ingested_at"]
pdf["source"] = "bronze/municipio.parquet"
pdf["version"] = "1.0"

# ---------- 3. DATA QUALITY ----------
# 3.1 Completude — chaves compostas sem nulos (crítica)
chaves = ["ano", "id_municipio", "serie", "rede"]
nulos_chave = pdf[chaves].isna().sum().sum()
print(f"[DQ] Nulos nas chaves: {nulos_chave}")

# 3.2 Unicidade — chave composta validada (0 duplicados no dado real)
dups = pdf.duplicated(subset=chaves).sum()
print(f"[DQ] Duplicados na chave composta: {dups}")

# 3.3 Domínio — rede dentro do esperado
dominios = set(pdf["rede"].astype(str).unique())
print(f"[DQ] Valores de rede: {dominios}")

# 3.4 Completude — proporcao_nivel nulos são ausência legítima (preservar)
nulos_prop = pdf["proporcao_aluno_nivel_0"].isna().sum()
print(f"[DQ] Nulos em proporcao_nivel_0 (legítimos): {nulos_prop}")

# ---------- 4. REGISTRAR DQ NO MONITORAMENTO ----------
# (ajuste o caminho/schema conforme o padrão do grupo)
# spark.sql("CREATE DATABASE IF NOT EXISTS monitoring")
# spark.sql("""
#   CREATE TABLE IF NOT EXISTS monitoring.dq_results (
#     table_name STRING, rule STRING, status STRING,
#     records_checked BIGINT, failures BIGINT, run_at TIMESTAMP
#   ) USING DELTA
# """)

# ---------- 5. GRAVAR EM DELTA ----------
df = spark.createDataFrame(pdf)
df = df.drop("_ingested_at", "_source_table")

# Caminho Delta (ajuste conforme o padrão do grupo)
delta_path = "/FileStore/silver/indicador_municipio"
df.write.mode("overwrite").format("delta").partitionBy("ano").save(delta_path)

# Criar tabela gerenciada para consulta
spark.sql(f"CREATE TABLE IF NOT EXISTS silver.indicador_municipio USING DELTA LOCATION '{delta_path}'")

print(f"\n[OK] Silver indicador_municipio gravada: {delta_path}")
print(f"     Registros: {df.count()} | Partições: {df.select('ano').distinct().count()}")